# 03) Integrate Bundle

Create an integration bundle that can be ingested into the strategy registry.
This notebook packages all backtest artifacts for downstream integration.

In [ ]:
import json
from pathlib import Path
from datetime import datetime

IMPORT_DIR = Path("imports")
IMPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = Path("exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load backtest results

Load from previous notebook or existing export files.

In [ ]:
# Option A: Use variables from notebook 02 (if in same session)
# backtest_id, result, payload already defined

# Option B: Load from files
# backtest_files = list(EXPORT_DIR.glob("backtest_report_*.json"))
# if backtest_files:
#     with open(backtest_files[-1], "r", encoding="utf-8") as f:
#         result = json.load(f)

## 2. Create integration bundle

Bundle schema includes all necessary artifacts for strategy integration.

In [ ]:
backtest_id = result.get("backtest_id", "unknown")

bundle = {
    "bundle_version": "1.0",
    "backtest_id": backtest_id,
    "strategy_id": result.get("strategy_id"),
    "payload": payload,
    "metrics_summary": result.get("metrics", {}),
    "artifact_paths": {
        "equity_curve_csv": f"exports/equity_curve_{backtest_id}.csv",
        "trades_csv": f"exports/trades_{backtest_id}.csv",
        "report_json": f"exports/backtest_report_{backtest_id}.json",
        "config_json": f"exports/config_{backtest_id}.json",
        "chart_png": f"exports/equity_curve_{backtest_id}.png",
    },
    "created_at": datetime.utcnow().isoformat() + "Z",
}

bundle_path = EXPORT_DIR / f"integration_bundle_{backtest_id}.json"
with open(bundle_path, "w", encoding="utf-8") as f:
    json.dump(bundle, f, ensure_ascii=False, indent=2)

print(f"Created bundle: {bundle_path}")

## 3. Validate bundle structure

Ensure all expected files exist before integration.

In [ ]:
missing = []
for path in bundle["artifact_paths"].values():
    if not Path(path).exists():
        missing.append(path)

if missing:
    print(f"⚠️ Missing artifacts: {missing}")
else:
    print("✅ All artifacts present")
    print(f"Metrics ready for integration: total_return={bundle['metrics_summary'].get('total_return')}")

## 4. Next steps

1. Bundle can be ingested via future integration endpoint (TBD)
2. Charts and data can be archived for research documentation
3. Metrics feed into strategy selection process